In [25]:
import pandas as pd

df = pd.read_csv('amazon_delivery.csv')
df

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43734,jlxf819993117,30,4.8,26.902328,75.794257,26.912328,75.804257,2022-03-24,11:35:00,11:45:00,Windy,High,motorcycle,Metropolitian,160,Home
43735,aevx342135787,21,4.6,0.000000,0.000000,0.070000,0.070000,2022-02-16,19:55:00,20:10:00,Windy,Jam,motorcycle,Metropolitian,180,Jewelry
43736,xnek760674819,30,4.9,13.022394,80.242439,13.052394,80.272439,2022-03-11,23:50:00,00:05:00,Cloudy,Low,scooter,Metropolitian,80,Home
43737,cynl434665991,20,4.7,11.001753,76.986241,11.041753,77.026241,2022-03-07,13:35:00,13:40:00,Cloudy,High,motorcycle,Metropolitian,130,Kitchen


In [26]:
df = df.drop(['Order_ID', 'Pickup_Time'], axis=1)

In [27]:
import datetime as dt
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

df['day_of_week'] = df['Order_Date'].dt.dayofweek

df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df['month'] = df['Order_Date'].dt.month

df['Order_Time'] = pd.to_datetime(df['Order_Time'], format= '%H:%M:%S', errors='coerce')

df['hour'] = df['Order_Time'].dt.hour

df['order_hour_block'] = pd.cut(df['hour'], bins=[0, 6, 12, 18, 24], labels=('night', 'morning', 'afternoon', 'evening'))

df['is_fast_delivery'] = (df['Delivery_Time'] >= df['Delivery_Time'].median()).astype(int) 

## The columns 'Order_Date', and 'Date_time' served their purpose. Having successfuly extracted the needed information from those columns, I can drop them without hesitation.

In [28]:
df = df.drop(['Order_Date', 'Order_Time', 'hour', 'Delivery_Time'], axis=1)

In [29]:
df.head()

,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Weather,Traffic,Vehicle,Area,Category,day_of_week,is_weekend,month,order_hour_block,is_fast_delivery
0,37,4.9,22.745049,75.892471,22.765049,75.912471,Sunny,High,motorcycle,Urban,Clothing,5,1,3,morning,0
1,34,4.5,12.913041,77.683237,13.043041,77.813237,Stormy,Jam,scooter,Metropolitian,Electronics,4,0,3,evening,1
2,23,4.4,12.914264,77.678400,12.924264,77.688400,Sandstorms,Low,motorcycle,Urban,Sports,5,1,3,morning,1
3,38,4.7,11.003669,76.976494,11.053669,77.026494,Sunny,Medium,motorcycle,Metropolitian,Cosmetics,1,0,4,afternoon,0
4,32,4.6,12.972793,80.249982,13.012793,80.289982,Cloudy,High,scooter,Metropolitian,Toys,5,1,3,afternoon,1


In [30]:
df.nunique()

Agent_Age             22
Agent_Rating          28
Store_Latitude       521
Store_Longitude      415
Drop_Latitude       4367
Drop_Longitude      4367
Weather                6
Traffic                5
Vehicle                4
Area                   4
Category              16
day_of_week            7
is_weekend             2
month                  3
order_hour_block       3
is_fast_delivery       2
dtype: int64

In [31]:
onehot_cols =['Weather', 'Traffic', 'Vehicle', 'Area', 'order_hour_block']
target_cols = ['Category']

df.shape

(43739, 16)

In [32]:
from sklearn.model_selection import train_test_split

X = df.drop('is_fast_delivery', axis=1)
y = df['is_fast_delivery']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

preprocessor = ColumnTransformer(transformers=[
    ('onehot', OneHotEncoder(), onehot_cols), 
    ('target', TargetEncoder(), target_cols)
], remainder='passthrough')

In [34]:
X_train_prepped = preprocessor.fit_transform(X_train, y_train)
X_test_prepped = preprocessor.transform(X_test)

In [35]:
X_train_prepped.shape

(34991, 34)

In [40]:
from xgboost import XGBClassifier

clf = XGBClassifier(random_state=42, n_jobs =-1)
clf.fit(X_train_prepped, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [41]:
train_score = clf.score(X_train_prepped, y_train)
test_score = clf.score(X_test_prepped, y_test)

print(f'train score: {train_score}')
print(f'test score: {test_score}')

train score: 0.9009745363093367
test score: 0.8211019661636946


In [42]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

params = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1],  
    'reg_lambda': [0, 2, 3, 5], 
    'reg_alpha': [0, 2, 3, 5], 
    'max_depth': [3, 5, 7]
}
boosted_clf = HalvingRandomSearchCV(
    estimator=clf,cv=5,verbose=2, param_distributions=params, 
)
boosted_clf.fit(X_train_prepped, y_train)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 7
min_resources_: 20
max_resources_: 34991
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 192
n_resources: 20
Fitting 5 folds for each of 192 candidates, totalling 960 fits
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=0; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=0; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=0; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=0; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=0; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=2; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=2; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=2; total time=   0.0s
[CV] END learning_rate=0.01, max_

C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 192 is smaller than n_iter=1749. Running 192 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=2; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=3; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=3; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=3; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=3; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=3; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=5; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=5; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=5; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=5; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, reg_alpha=0, reg_lambda=5; total time=   0.0s
[CV] END learning_rate=0.01, max

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.01, 0.03, ...], 'max_depth': [3, 5, ...], 'reg_alpha': [0, 2, ...], 'reg_lambda': [0, 2, ...]}"
,"n_candidates n_candidates: ""exhaust"" or int, default=""exhaust""The number of candidate parameters to sample, at the firstiteration. Using 'exhaust' will sample enough candidates so that thelast iteration uses as many resources as possible, based on`min_resources`, `max_resources` and `factor`. In this case,`min_resources` cannot be 'exhaust'.",'exhaust'
,"factor factor: int or float, default=3The 'halving' parameter, which determines the proportion of candidatesthat are selected for each subsequent iteration. For example,``factor=3`` means that only one third of the candidates are selected.",3
,"resource resource: ``'n_samples'`` or str, default='n_samples'Defines the resource that increases with each iteration. By default,the resource is the number of samples. It can also be set to anyparameter of the base estimator that accepts positive integervalues, e.g. 'n_iterations' or 'n_estimators' for a gradientboosting estimator. In this case ``max_resources`` cannot be 'auto'and must be set explicitly.",'n_samples'
,"max_resources max_resources: int, default='auto'The maximum number of resources that any candidate is allowed to usefor a given iteration. By default, this is set ``n_samples`` when``resource='n_samples'`` (default), else an error is raised.",'auto'
,"min_resources min_resources: {'exhaust', 'smallest'} or int, default='smallest'The minimum amount of resource that any candidate is allowed to usefor a given iteration. Equivalently, this defines the amount ofresources `r0` that are allocated for each candidate at the firstiteration.- 'smallest' is a heuristic that sets `r0` to a small value: - ``n_splits * 2`` when ``resource='n_samples'`` for a regression problem - ``n_classes * n_splits * 2`` when ``resource='n_samples'`` for a classification problem - ``1`` when ``resource != 'n_samples'``- 'exhaust' will set `r0` such that the **last** iteration uses as much resources as possible. Namely, the last iteration will use the highest value smaller than ``max_resources`` that is a multiple of both ``min_resources`` and ``factor``. In general, using 'exhaust' leads to a more accurate estimator, but is slightly more time consuming. 'exhaust' isn't available when `n_candidates='exhaust'`.Note that the amount of resources used at each iteration is always amultiple of ``min_resources``.",'smallest'
,"aggressive_elimination aggressive_elimination: bool, default=FalseThis is only relevant in cases where there isn't enough resources toreduce the remaining candidates to at most `factor` after the lastiteration. If ``True``, then the search process will 'replay' thefirst iteration for as long as needed until the number of candidatesis small enough. This is ``False`` by default, which means that thelast iteration may evaluate more than ``factor`` candidates. See:ref:`aggressive_elimination` for more details.",False
,"cv cv: int, cross-validation generator or an iterable, default=5Determines the cross-validation splitting strategy.Possible inputs for cv are:- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y

In [43]:
best_model = boosted_clf.best_estimator_

best_model_train_score = best_model.score(X_train_prepped, y_train)
best_model_test_score = best_model.score(X_test_prepped, y_test)
 
print(f'train score: {best_model_train_score}')
print(f'test score: {best_model_test_score}')

train score: 0.8172958760824212
test score: 0.8102423411065386


In [45]:
print(boosted_clf.best_params_)

{'reg_lambda': 2, 'reg_alpha': 0, 'max_depth': 3, 'learning_rate': 0.1}
